# 3 gNB / 1 eMBB UE Mobility Radio Trace

This notebook validates the radio, scheduler, fading, and mobility timescale with four one-UE simulations.

- 3 same-carrier gNBs
- 1 moving eMBB UE per simulation
- 4 different initial positions and trajectories
- `step_dt = mobility_dt = 1e-3` seconds
- `radio_substeps = 1`

One recorded row is one coherent 1 ms tick: mobility update, traffic generation, fading-column advance, scheduler allocation, PRB usage, and service.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env

plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.25})
np.set_printoptions(precision=3, suppress=True)

PROJECT_ROOT

## Configuration

In [ ]:
SEED = 20260617
SIM_DURATION_S = 5 * 60
RADIO_DT_S = 1e-3
MOBILITY_DT_S = RADIO_DT_S
RADIO_SUBSTEPS = 1
N_RADIO_TICKS = int(SIM_DURATION_S / RADIO_DT_S)
PLOT_MAX_POINTS_PER_SCENARIO = 5000
SLICE_TYPE = "eMBB"
TOTAL_PRBS_PER_GNB = 100
SLICE_PRB_BUDGETS = {"eMBB": 50, "URLLC": 30, "mMTC": 20}  # ignored by the old full-gNB scheduler path
MAX_PRBS_PER_UE = 30  # ignored by the old full-gNB scheduler path

gnb_configs = [
    {"id": 0, "x": 0.0, "y": 0.0, "coverage_radius": 520.0, "carrier_id": 0, "n_prbs": TOTAL_PRBS_PER_GNB},
    {"id": 1, "x": 450.0, "y": 0.0, "coverage_radius": 520.0, "carrier_id": 0, "n_prbs": TOTAL_PRBS_PER_GNB},
    {"id": 2, "x": 225.0, "y": 390.0, "coverage_radius": 520.0, "carrier_id": 0, "n_prbs": TOTAL_PRBS_PER_GNB},
]

SCENARIOS = [
    {"name": "g0_center_to_overlap", "x": 80.0, "y": 25.0, "vx": 1.20, "vy": 0.35},
    {"name": "g1_edge_to_center", "x": 415.0, "y": 40.0, "vx": -1.10, "vy": 0.20},
    {"name": "g2_down_to_middle", "x": 225.0, "y": 335.0, "vx": 0.0, "vy": -1.25},
    {"name": "central_diagonal", "x": 230.0, "y": 115.0, "vx": 0.75, "vy": 0.95},
]

assert RADIO_SUBSTEPS == 1
assert RADIO_DT_S == MOBILITY_DT_S

print(f"Simulating {SIM_DURATION_S} seconds = {N_RADIO_TICKS:,} coherent 1 ms ticks per scenario")
pd.DataFrame(SCENARIOS)

## Helpers

In [ ]:
def dbm_to_watts(dbm):
    return 10.0 ** ((float(dbm) - 30.0) / 10.0)

def watts_to_dbm(watts):
    watts = max(float(watts), 1e-30)
    return 10.0 * np.log10(watts) + 30.0

def finite_scalar(value):
    try:
        value = float(value)
    except Exception:
        return np.nan
    return value if np.isfinite(value) else np.nan

def make_env(seed):
    rng = np.random.default_rng(seed)
    env = create_multignb_env(
        rng=rng,
        n=4,
        gnb_configs=gnb_configs,
        slots_per_step=5,
        L1_level=False,
        step_dt=RADIO_DT_S,
        mobility_dt=MOBILITY_DT_S,
        radio_substeps=RADIO_SUBSTEPS,
        max_episode_steps=N_RADIO_TICKS + 5,
        slice_prb_budgets=SLICE_PRB_BUDGETS,
        max_prbs_per_ue=MAX_PRBS_PER_UE,
    )
    env.reset(seed=seed)
    return env

def fading_state_for_ue(env, ue_id):
    state = env._fading_users.get(int(ue_id), {})
    return {
        "fading_trace_idx": finite_scalar(state.get("trace_idx", np.nan)),
        "fading_column_idx": finite_scalar(state.get("index", np.nan)),
        "fading_step_direction": finite_scalar(state.get("step", np.nan)),
        "fading_n_columns": finite_scalar(state.get("n_cols", np.nan)),
    }

def sample_for_plot(group, max_points=PLOT_MAX_POINTS_PER_SCENARIO):
    if len(group) <= max_points:
        return group
    step = int(np.ceil(len(group) / max_points))
    sampled = group.iloc[::step].copy()
    if sampled.index[-1] != group.index[-1]:
        sampled = pd.concat([sampled, group.tail(1)], axis=0)
    return sampled

def collect_row(env, ue_id, scenario_name, tick, info):
    ue = env.get_ue(ue_id)
    radio = env.get_ue_radio_metrics(ue_id)
    serving = -1 if radio["serving_gnb"] is None else int(radio["serving_gnb"])
    kpi = info["slice_kpis"].get((serving, SLICE_TYPE), {}) if serving >= 0 else {}
    budget_prbs = int(kpi.get("budget_prbs", env.get_slice_prb_budget(serving, SLICE_TYPE) if serving >= 0 else 1))
    row = {
        "scenario": scenario_name,
        "tick": int(tick),
        "time_s": float(tick) * RADIO_DT_S,
        "time_ms": float(tick) * RADIO_DT_S * 1000.0,
        "radio_dt_s": float(info["radio_dt"]),
        "mobility_dt_s": float(info["mobility_dt"]),
        "radio_substeps": int(info["radio_substeps"]),
        "ue_id": int(ue_id),
        "x": float(ue.x),
        "y": float(ue.y),
        "vx": float(ue.vx),
        "vy": float(ue.vy),
        "serving_gnb": serving,
        "connected": bool(radio["connected"]),
        "sinr_db": float(radio["sinr_db"]),
        "scheduled_sinr_db": float(radio["scheduled_sinr_db"]),
        "snr_db": float(radio["snr_db"]),
        "rsrp_dbm": float(radio["rsrp_dbm"]),
        "rssi_dbm": float(radio["rssi_dbm"]),
        "rsrq_db": float(radio["rsrq_db"]),
        "rx_power_dbm": float(radio["rx_power_dbm"]),
        "interference_dbm": float(radio["interference_dbm"]),
        "noise_dbm": float(radio["noise_dbm"]),
        "mcs": int(radio["mcs"]),
        "spectral_efficiency": float(radio["spectral_efficiency"]),
        "rx_probability": float(radio["rx_probability"]),
        "allocated_prbs": int(radio["allocated_prbs"]),
        "scheduled_bits": float(radio["scheduled_bits"]),
        "served_bits": float(radio["served_bits"]),
        "new_bits": float(radio["new_bits"]),
        "queue_bits": float(radio["queue"]),
        "throughput_bps": float(radio["throughput"]),
        "delay_steps": float(radio["delay_steps"]),
        "traffic_packet_size_bits": float(radio["traffic_packet_size_bits"]),
        "offered_bit_rate": float(radio["offered_bit_rate"]),
        "slice_budget_prbs": budget_prbs,
        "slice_used_prbs": int(kpi.get("used_prbs", 0)),
        "slice_demand_prbs": int(kpi.get("demand_prbs", 0)),
        "slice_load": float(kpi.get("load", 0.0)),
        "slice_queue_pressure": float(kpi.get("queue_pressure", 0.0)),
        "slice_tx_success_ratio": float(kpi.get("tx_success_ratio", 0.0)),
    }
    row.update(fading_state_for_ue(env, ue_id))

    for gnb in env.gnbs:
        gnb_id = int(gnb.id)
        metrics = env._compute_link_metrics(gnb, ue)
        row[f"g{gnb_id}_sinr_db"] = float(metrics["sinr_db"])
        row[f"g{gnb_id}_rx_power_dbm"] = float(metrics["rx_power_dbm"])
        row[f"g{gnb_id}_rsrp_dbm"] = float(metrics["rsrp_dbm"])
        row[f"g{gnb_id}_rssi_dbm"] = float(metrics["rssi_dbm"])
        row[f"g{gnb_id}_rsrq_db"] = float(metrics["rsrq_db"])
        row[f"g{gnb_id}_snr_db"] = float(metrics["snr_db"])

    return row

def run_one_scenario(spec, idx):
    env = make_env(SEED + idx)
    ue_id = env.add_ue(
        x=spec["x"],
        y=spec["y"],
        vx=spec["vx"],
        vy=spec["vy"],
        slice_type=SLICE_TYPE,
        bit_rate=2_000_000.0,
        packet_size_bits=1000.0,
        traffic_model="fixed_packet_cbr",
    )

    rows = []
    for tick in range(1, N_RADIO_TICKS + 1):
        _obs, _reward, terminated, truncated, info = env.step(0)
        rows.append(collect_row(env, ue_id, spec["name"], tick, info))
        if terminated or truncated:
            break

    trace = pd.DataFrame(rows)
    radio_keys = sorted(env.get_ue_radio_metrics(ue_id).keys())
    slice_keys = sorted(info["slice_kpis"][(int(trace["serving_gnb"].iloc[-1]), SLICE_TYPE)].keys())
    env.close()
    return trace, radio_keys, slice_keys

## Run Four Mobility Simulations

In [ ]:
traces = []
radio_metric_keys = None
slice_metric_keys = None

for idx, spec in enumerate(SCENARIOS):
    trace_i, radio_metric_keys, slice_metric_keys = run_one_scenario(spec, idx)
    traces.append(trace_i)

trace = pd.concat(traces, ignore_index=True)

display(trace.head(12))
display(trace.groupby("scenario").tail(3))
print("Rows:", len(trace))
print("Radio metrics exposed by get_ue_radio_metrics():")
print(radio_metric_keys)
print("Slice KPI metrics exposed by info['slice_kpis']:")
print(slice_metric_keys)

## Coherent Timescale Checks

In [ ]:
assert (trace["radio_dt_s"] == RADIO_DT_S).all()
assert (trace["mobility_dt_s"] == MOBILITY_DT_S).all()
assert (trace["radio_substeps"] == 1).all()

for scenario, group in trace.groupby("scenario"):
    assert np.allclose(group["time_ms"].diff().dropna(), RADIO_DT_S * 1000.0), scenario
    assert (group["allocated_prbs"] == group["slice_used_prbs"]).all(), scenario
    normal_fading_steps = group["fading_column_idx"].diff().dropna().abs()
    normal_fading_steps = normal_fading_steps[normal_fading_steps < 100]
    assert (normal_fading_steps == 1).mean() > 0.95, scenario

summary = trace.groupby("scenario").agg(
    rows=("tick", "count"),
    time_span_ms=("time_ms", "max"),
    start_x=("x", "first"),
    start_y=("y", "first"),
    end_x=("x", "last"),
    end_y=("y", "last"),
    mean_sinr_db=("sinr_db", "mean"),
    min_sinr_db=("sinr_db", "min"),
    max_sinr_db=("sinr_db", "max"),
    mean_scheduled_sinr_db=("scheduled_sinr_db", "mean"),
    mean_rsrp_dbm=("rsrp_dbm", "mean"),
    mean_rsrq_db=("rsrq_db", "mean"),
    max_allocated_prbs=("allocated_prbs", "max"),
    mean_allocated_prbs=("allocated_prbs", "mean"),
    total_served_bits=("served_bits", "sum"),
    max_queue_bits=("queue_bits", "max"),
)

display(summary)
print("Coherent timing checks passed")

## Topology Before And After

In [ ]:
def draw_topology(ax, group, title):
    colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
    for cfg in gnb_configs:
        color = colors[int(cfg["id"])]
        circle = plt.Circle((cfg["x"], cfg["y"]), cfg["coverage_radius"], fill=False, color=color, alpha=0.25, linewidth=1.4)
        ax.add_patch(circle)
        ax.scatter(cfg["x"], cfg["y"], s=120, marker="^", color=color, label=f"gNB {cfg['id']}")
        ax.text(cfg["x"] + 10, cfg["y"] + 10, f"g{cfg['id']}", color=color, weight="bold")

    plot_group = sample_for_plot(group)
    ax.plot(plot_group["x"], plot_group["y"], color="black", linewidth=1.5, alpha=0.7, label="UE path")
    ax.scatter(group["x"].iloc[0], group["y"].iloc[0], s=90, color="white", edgecolor="black", linewidth=1.8, label="UE start")
    ax.scatter(group["x"].iloc[-1], group["y"].iloc[-1], s=90, color="red", edgecolor="black", linewidth=1.0, label="UE end")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-120, 620)
    ax.set_ylim(-120, 570)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (scenario, group) in zip(axes.ravel(), trace.groupby("scenario", sort=False)):
    draw_topology(ax, group, scenario)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=6)
fig.tight_layout(rect=(0, 0, 1, 0.95))

## SINR Over Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
for ax, (scenario, group) in zip(axes.ravel(), trace.groupby("scenario", sort=False)):
    plot_group = sample_for_plot(group)
    ax.plot(plot_group["time_ms"], plot_group["sinr_db"], label="serving nominal SINR", linewidth=2)
    ax.plot(plot_group["time_ms"], plot_group["scheduled_sinr_db"], label="scheduled fading mean SINR", alpha=0.85)
    for gnb_id in range(3):
        ax.plot(plot_group["time_ms"], plot_group[f"g{gnb_id}_sinr_db"], linestyle="--", alpha=0.55, label=f"g{gnb_id} candidate SINR")
    ax.set_title(scenario)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("SINR (dB)")

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
fig.tight_layout(rect=(0, 0, 1, 0.88))

## Radio And Service Metrics

In [ ]:
metric_specs = [
    ("rsrp_dbm", "RSRP (dBm)"),
    ("rsrq_db", "RSRQ (dB)"),
    ("rx_probability", "RX probability"),
    ("allocated_prbs", "allocated PRBs"),
    ("scheduled_bits", "scheduled bits"),
    ("served_bits", "served bits"),
    ("queue_bits", "queue bits"),
    ("throughput_bps", "throughput (bps)"),
    ("mcs", "MCS"),
    ("slice_load", "eMBB slice load"),
]

fig, axes = plt.subplots(len(metric_specs), 1, figsize=(14, 24), sharex=True)
for ax, (key, label) in zip(axes, metric_specs):
    for scenario, group in trace.groupby("scenario", sort=False):
        plot_group = sample_for_plot(group)
        ax.plot(plot_group["time_ms"], plot_group[key], label=scenario, linewidth=1.5)
    ax.set_ylabel(label)

axes[-1].set_xlabel("time (ms)")
axes[0].legend(loc="upper center", ncol=2)
fig.tight_layout()

## Available Metrics Table

In [ ]:
available_columns = pd.DataFrame({"trace_column": sorted(trace.columns)})
display(available_columns)

numeric_summary = trace.select_dtypes(include=[np.number]).groupby(trace["scenario"]).agg(["min", "mean", "max"])
display(numeric_summary)

## Save Trace CSV

In [ ]:
out_dir = PROJECT_ROOT / "results" / "three_gnb_one_embb_coherent_radio_trace"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "mobility_radio_trace_4_scenarios.csv"
trace.to_csv(csv_path, index=False)
csv_path